<!-- # Title here {-} -->

Per Halvorsen   \
[pmhalvor@uio.no](mailto:pmhalvor@uio.no) \
GEO4460         

# Introduction


In this report, we will build a watershed model for a region of Vestland, Norway, using ArcGIS Pro. 
The main data source will be the N50 Kartdata, which provides both elevation and land cover information.
We'll apply different thresholds on flow accumulation in an attempt to delineate the watershed boundaries similar to those present in the N50 Kartdata.

Topics we will explore:

- Filling sinks in the domain elevation model
- Calculating flow direction and flow accumulation
- Mask filtering with true and false values from rasters (or constant values)
- Delineating streams and watersheds from flow layers
- Comparing results with N50 Kartdata



# Data

Elevation models for Norway are readily available from the Norwegian Mapping Authority (Kartverket). [N50 DTM](https://kartkatalog.geonorge.no/metadata/dtm-50/e25d0104-0858-4d06-bba8-d154514c11d2) can be downloaded from the online catalog for specified cells across the country. 
If a user is interested in large areas, spanning multiple cells, the downloaded data can easily be merged into a single raster using ArcGIS Pro. 

The [N50 Kartdata](https://kartkatalog.geonorge.no/metadata/n50-kartdata/ea192681-d039-42ec-b1bc-f3ce04c189ac) is available from the same catalog. These data will be used to compare the resulting stream layers against, evaluating the thresholds we applied for the flow accumulation filtering. 

The N50 Kartdata also contains information on lakes, which we will incorperate in our watershed model, to give a more hydrologically realistic representation of the watershed.

# Methods

In class, we saw that the typical process for hydrological modeling is as follows:
![Hydrological modeling process](img/hydro_model.png)
Figure 1: Typical hydrological modeling process. Source: [GEO4460 lecture notes](https://www.uio.no/studier/emner/matnat/geofag/GEO4460/).

For this lab exercise, we recreated most of these steps, focusing mainly on the flow accumulation and stream delineation steps.

Specifically, we:

- Filled sinks in the elevation model
- Calculated flow direction and flow accumulation
- Applied a mask to the flow accumulation layer to filter out low values
- (Optional) Subtracted the lakes from the flow accumulation layer to remove them from stream delineation
- Built a stream-order layer from the different filtered flow accumulation layers
    - The results here were compared against the N50 Kartdata stream layer to ensure our workflow was producing reasonable results
- Converted the streams to features and stream links, connecting smaller, lower order streams to larger, higher order streams or main rivers
- Calculated the watershed boundaries using the stream links and flow direction layers

The steps were configured through a model in ArcGIS Pro, allowing for easy adjustments and re-runs, which was particularly useful for testing different flow accumulation thresholds.

![Model in ArcGIS Pro](img/workflow.png)
Firgure 2: Our model workflow in ArcGIS Pro to build the watershed model. 

## Fill sinks and calculate flow
This report assumes we are using a precomputed and merged digital elevation model (DEM) of the area. 
We filled sinks in the DEM to ensure that all flow paths are directed towards lower elevations. 
This is crucial for accurate flow direction and accumulation calculations.

With the filled DEM, we calculated the flow direction using the **Flow Direction** tool in ArcGIS Pro.
This tool computes the direction of flow for each cell in the raster, allowing us to understand how water would flow across the landscape.
This means each cells is categorized with a direction: north, northeast, east, southeast, south, southwest, west, northwest, or no flow.

Next, we calculated the flow accumulation using the **Flow Accumulation** tool.
This tool counts the number of upstream cells that contribute flow to each cell, resulting in a raster where each cell's value represents the total amount of flow that would pass through it.
This tells us how much water would accumulate at each point in the landscape, which influences how large streams and rivers would become.

## Filtering lakes and low flow accumulation values
Given that lakes are basically large accumulation areas, we can remove these from our flow accumulation and flow direction layers, labeling them as "no flow" areas.
To remove lakes from the flow accumulation layer, we used the **Extract by Mask** tool in ArcGIS Pro.
This tool allows us to extract the flow accumulation values that fall within the lake polygons, effectively masking out the lakes from the flow accumulation raster.
We then set the flow accumulation values within the lake polygons to zero, indicating that no flow accumulates in these areas.
(This step is optional. We can also choose to keep the lakes in the flow accumulation layer. More on each of these options in the results section.)


To filter away low flow accumulation areas, we used the **Con** tool in ArcGIS Pro, which allows us to apply a conditional statement to our flow accumulation raster.
We set the condition to check if the cell value is greater than a specified threshold (e.g., 1000), and if true, we keep the original flow accumulation value; otherwise, we set it to zero.

We experimented with three thresholds: 500, 1000, and 10000.
These numbers were selected based on the values from the flow accumulation raster, which ranged from 0 to 26000. 
The max values in that raster were clearly large rivers/fjord outlets, so the thresholds were set considerably lower to capture more of the smaller streams. 

## Stream delineation and stream order
With the filtered flow accumulation raster, we delineated streams using the **Stream Order** tool in ArcGIS Pro.
This tool allows us to assign a stream order to each cell based on the flow accumulation values.
Low order streams are small streams with low flow accumulation, while high order streams are larger rivers with high flow accumulation.

We used the **Stream Link** tool to convert the stream order raster into a vector layer, which connects smaller streams to larger ones.
This step helped build a network of streams, where each stream segment is linked to its upstream and downstream neighbors.
<!-- We then used the **Stream to Feature** tool to convert the stream links into a feature layer, which allows us to visualize the streams on a map. -->

## Watershed Raster
Finally, we used the **Watershed** tool to delineate the watershed boundaries based on the stream links and flow direction layers.
The inputs to this layer were the stream links and the flow direction raster.
This tool identifies the area that drains into each stream segment, effectively delineating the watershed boundaries.



# Results

## Stream Orders 
![Stream orders](img/stream_order_all.png)
Figure 3: Stream orders for different flow accumulation thresholds (left to right: 0, 500, 1000, 10000). As the threshold increases, the number of streams remaining decreases, and the remaining streams become larger and more significant. For larger stream order images, see [Appendix A](#appendix-a).


## Watershed Raster
![Watershed raster](img/watershed.png)
Figure 4: Final watershed raster, showing the delineated watersheds based on the stream links and flow direction layers. Each color represents a different watershed area. Overlayed is the the streams layer generated from the 1000-thresholded stream links. 


# Discussion 

## Comparing Stream Orders

We compared stream orders with 4 different filter thresholds: 0 (unfiltered), 500, 1k, and 10k.  
The **unfiltered** stream order map shows many streams scattered across the entire area observed.

The **500** and **1k** filters retain a moderate number of streams. These seem to provide the most meaningful results, as they show streams along many of the main rivers, along with some smaller streams leading into the main rivers.  
The **10k** filter shows only the largest fjord outlets, which likely means the threshold is too high.

## Comparing with N50 Kartdata
![N50 Kartdata streams](img/comparison.png)  
Figure 5: Comparison of our stream order results with the N50 Kartdata streams. The left image shows the N50 Kartdata streams, while the right image shows our stream order results with a 1k threshold.

In the overlay comparison, we see the N50 Kartdata rivers in blue and our stream order results in purple, yellow, and red (orders 1, 2, and 3, respectively).  
All of the rivers in the N50 data are covered by streams generated from our analysis, mostly of orders 1 and 2 (purple and yellow).

Our streams actually look more connected and continuous than the N50 Kartdata streams.  
This is an interesting finding, since the DEM used for the flow accumulation and stream order calculations is based on the same elevation data present in the N50 Kartdata.

Comparing our found streams to the underlying base map, however, our streams appear shorter than most river lines.  
This shows that it can often be beneficial to source multiple datasets when conducting an analysis, to ensure that the features generated are the best possible representation of the underlying data.

If we lowered our threshold even further, we might be able to cover all the streams and rivers present in the underlying base map.  
This would be easy to test and tune with our model, as we can simply change the threshold value and re-run the model to see how the results change.  
We will, however, save that for future work, as the current results already achieve the goal outlined in the introduction.

## Analysis of the Watershed Raster
![Watershed raster](img/watershed.png)
Figure 6: Watershed raster, showing the delineated watersheds based on the stream links and flow direction layers. Each color represents a different watershed area.

Reviewing the watershed raster, we see that each color represents an area that feeds into a stream or river. 
The partitioning between each watershed area seems reasonable compared to the underlying base map.
The watershed boundaries appear to follow the topography of the landscape, with higher elevations forming natural divides between watersheds.


# Summary

In this report, we successfully built a watershed model for a region of Vestland, Norway, using ArcGIS Pro and the N50 Kartdata.
We examined different thresholds for flow accumulation filtering, observing the effects of over- and under-filtering the flow accumulation layer.
We found that a threshold of 1000 provided the best results, capturing most of the streams and rivers present in the N50 Kartdata.

The model we built allows for easy adjustments and re-runs, making it a flexible tool for hydrological modeling.
The results show that our workflow produces reasonable stream delineation and watershed boundaries, which can be further refined by adjusting the flow accumulation thresholds.
The watershed model can be used for various applications, such as hydrological analysis, water resource management, and environmental monitoring.

# References

- N50 Kartdata. (n.d.). Retrieved from [https://kartkatalog.geonorge.no/metadata/n50-kartdata/ea192681-d039-42ec-b1bc-f3ce04c189ac](https://kartkatalog.geonorge.no/metadata/n50-kartdata/ea192681-d039-42ec-b1bc-f3ce04c189ac)
- Norwegian Mapping Authority. (n.d.). N50 DTM. Retrieved from [https://kartkatalog.geonorge.no/metadata/dtm-50/e25d0104-0858-4d06-bba8-d154514c11d2](https://kartkatalog.geonorge.no/metadata/dtm-50/e25d0104-0858-4d06-bba8-d154514c11d2)
- ArcGIS Pro. (n.d.). Retrieved from [https://www.esri.com/en-us/arcgis/products/arcgis-pro/overview](https://www.esri.com/en-us/arcgis/products/arcgis-pro/overview)

# Appendix A
## Stream Orders
![Stream orders 0](img/stream_order.png)
Figure 7: Stream order with no filter applied. This shows all streams, including very small ones.


![Stream orders 500](img/stream_order_500.png)  
Figure 8: Stream order with a 500 threshold applied. This shows a moderate amount of streams, including some smaller ones leading into larger rivers.


![Stream orders 1000](img/stream_order_1k.png)
Figure 9: Stream order with a 1000 threshold applied. This shows a smaller amount of streams, focusing on the larger ones leading into the main rivers.


![Stream orders 10000](img/stream_order_10k.png)
Figure 10: Stream order with a 10000 threshold applied. This shows only the largest streams, likely too high of a threshold for this area.